# AEDWIP Map

In [1]:
import ipynbname

# use display() to print an html version of a data frame
# useful if dataFrame output is not generated by last like of cell
from IPython.display import display

# import joblib
# import math
import numpy as np
import os
import pandas as pd
# pd.set_option('display.max_rows', None)

# from sklearn.ensemble        import RandomForestClassifier

import sys

notebookName = ipynbname.name()
notebookPath = ipynbname.path()
notebookDir = os.path.dirname(notebookPath)

import logging
loglevel = "INFO"
# loglevel = "WARN"
# logFMT = "%(asctime)s %(levelname)s [thr:%(threadName)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logFMT = "%(asctime)s %(levelname)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logging.basicConfig(format=logFMT, level=loglevel)    
logger = logging.getLogger(notebookName)

In [2]:
# setting the python path allows us to run python scripts from using
# the CLI. 
ORIG_PYTHONPATH = os.environ['PYTHONPATH']

#deconvolutionModules = notebookPath.parent.joinpath("../../deconvolutionAnalysis/python/")
# deconvolutionModules = notebookPath.parent.joinpath("../..")
deconvolutionModules = notebookPath.parent.joinpath("../../python")
print("deconvolutionModules: {}\n".format(deconvolutionModules))

PYTHONPATH = ORIG_PYTHONPATH + f':{deconvolutionModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

# intraExtraRNA_POCModules=notebookPath.parent.joinpath("../../intraExtraRNA_POC/python/src")
# intraExtraRNA_POCModules=notebookPath.parent.joinpath("/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/python/tempus/jupyterNotebooks/../../../../intraExtraRNA_POC/python/src")
intraExtraRNA_POCModules=notebookPath.parent.joinpath("../../../intraExtraRNA_POC/python/src")
print("intraExtraRNA_POCModules: {}\n".format(intraExtraRNA_POCModules))

PYTHONPATH = PYTHONPATH + f':{intraExtraRNA_POCModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

os.environ["PYTHONPATH"] = PYTHONPATH
PYTHONPATH = os.environ["PYTHONPATH"]
print("PYTHONPATH: {}\n".format(PYTHONPATH))

# to be able to import our local python files we need to set the sys.path
# https://stackoverflow.com/a/50155834
sys.path.append( str(deconvolutionModules) )
sys.path.append( str(intraExtraRNA_POCModules) )
print("\nsys.path:\n{}\n".format(sys.path))

deconvolutionModules: /private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/tempus/jupyterNotebooks/../../python

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/tempus/jupyterNotebooks/../../python

intraExtraRNA_POCModules: /private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/tempus/jupyterNotebooks/../../../intraExtraRNA_POC/python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/tempus/jupyterNotebooks/../../python:/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/tempus/jupyterNotebooks/../../../intraExtraRNA_POC/python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/tempus/jupyterNotebooks/../../python:/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/tempus/jupyterNotebooks/../../../intraExtraRNA_POC/python/src


sys.pat

In [3]:
# local imports
# from analysis.utilities import saveList
# from intraExtraRNA.elifeUtilities import loadElifeTrainingData
# from models.mlUtilities import saveLabelEncoder
from intraExtraRNA.elifeUtilities import selectFeatures

# Get Gene List

In [21]:
hackName = 'hack-2025-02-20'
dataDir = f'/private/groups/kimlab/aedavids/deconvolution/tempus/{hackName}'
files = ['top20FromBest500GTEx_TCGA.LUAD_vs_all.results',
         'top20FromBest500GTEx_TCGA.LUSC_vs_all.results',
         'top20FromBest500GTEx_TCGA.Lung_vs_all.results',
         'top20FromBest500GTEx_TCGA.Whole_Blood_vs_all.results' ]

outDir = f'{dataDir}/{notebookName}.out'
os.makedirs(outDir, exist_ok=True)
print(f'outDir:\n{outDir}')

outDir:
/private/groups/kimlab/aedavids/deconvolution/tempus/hack-2025-02-20/AEDWIP_MAP.out


In [5]:
f = files[0]
category = f.split(".")[1]
category

'LUAD_vs_all'

In [6]:
def loadResults():
    retDict = dict()
    for f in files:
        category = f.split(".")[1]
        fPath = f'{dataDir}/{f}'
        df = pd.read_csv(fPath)
        print(f'loaded {fPath}')
        retDict[category] = df

    return retDict

deseqResultsDict = loadResults()

loaded /private/groups/kimlab/aedavids/deconvolution/tempus/hack-2025-02-20/top20FromBest500GTEx_TCGA.LUAD_vs_all.results
loaded /private/groups/kimlab/aedavids/deconvolution/tempus/hack-2025-02-20/top20FromBest500GTEx_TCGA.LUSC_vs_all.results
loaded /private/groups/kimlab/aedavids/deconvolution/tempus/hack-2025-02-20/top20FromBest500GTEx_TCGA.Lung_vs_all.results
loaded /private/groups/kimlab/aedavids/deconvolution/tempus/hack-2025-02-20/top20FromBest500GTEx_TCGA.Whole_Blood_vs_all.results


In [7]:
deseqResultsDict.keys()

dict_keys(['LUAD_vs_all', 'LUSC_vs_all', 'Lung_vs_all', 'Whole_Blood_vs_all'])

In [8]:
aedwipDF = deseqResultsDict['LUAD_vs_all']
aedwipDF

,name,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
0,DES,89881.666167,-4.153133,0.180200,-23.047413,1.561467e-117,2.267881e-115
1,CLU,56070.788880,-2.273153,0.107877,-21.071755,1.444750e-98,1.413293e-96
2,KRT6A,36638.110998,-3.608496,0.236553,-15.254489,1.536931e-52,4.225783e-51
3,IGHA1,25987.995442,2.265104,0.173259,13.073548,4.663817e-39,7.564886e-38
4,CRYAB,21594.014674,-5.135958,0.127156,-40.390908,0.000000e+00,0.000000e+00
5,KRT17,19620.982687,-3.223883,0.188361,-17.115405,1.139206e-65,4.666331e-64
6,RGS5,18048.226869,-2.193606,0.118217,-18.555835,7.316229e-77,4.103218e-75
7,NDRG2,17790.572037,-2.249253,0.102673,-21.907055,2.225110e-106,2.565822e-104
8,SERPINA3,14604.186235,-2.506750,0.155628,-16.107276,2.267954e-58,7.491534e-57
9,SFTPB,14421.444640,6.019867,0.241647,24.911810,5.541549e-137,1.204986e-134


In [9]:
def getGeneSetDict(deseqResultsDict : pd.DataFrame) -> dict[str,pd.DataFrame] :
    retDict = dict()
    for category,resultsDF in deseqResultsDict.items():
        genesDF = resultsDF.loc[:, ['name']]
        retDict[category] = genesDF

    return retDict
        

geneSetDict = getGeneSetDict( deseqResultsDict)

In [10]:
geneSetDict['LUAD_vs_all']

,name
0,DES
1,CLU
2,KRT6A
3,IGHA1
4,CRYAB
5,KRT17
6,RGS5
7,NDRG2
8,SERPINA3
9,SFTPB


In [11]:
def createListOfGenes( geneSetDict : dict[str, pd.DataFrame] ) -> list[str] :
    retList = []
    for category, df in geneSetDict.items():
        g = df.loc[:, 'name'].to_list()
        retList = retList + g

    # use a set to get rid of duplicates
    return list( set(retList) )

listOfGenes = createListOfGenes(geneSetDict)

In [12]:
print( len(listOfGenes) )

60


# Create the mapping data frame

In [13]:
 elifeGenes, missingElifeGenes, mapDF = selectFeatures(listOfGenes)

2025-02-20 20:47:50,817 INFO intraExtraRNA.elifeUtilities selectFeatures() line:364] [BEGIN]
2025-02-20 20:47:50,818 INFO intraExtraRNA.elifeUtilities selectFeatures() line:365] [genes: ['APOD', 'CD24', 'MT-CYB', 'FTH1', 'FAM107A', 'FTL', 'MYH11', 'FABP5', 'SORBS1', 'MT-ND3', 'KIF5A', 'HBA1', 'MT-RNR1', 'FHL1', 'KRT5', 'LMOD1', 'C4A', 'SFTPA2', 'TPM2', 'DSP', 'PERP', 'LSU-rRNA', 'MT-CO3', 'MT-ND1', 'CLU', 'DES', 'SFTPB', 'A2M', 'PDK4', 'TRIM29', 'HBB', 'ATP1A2', 'SYNM', 'KRT17', 'MT-ND5', 'MT-ATP6', 'PKP1', 'FN1', 'RGS5', 'SERPINA3', 'APOE', 'MT-ND4', 'ADH1B', 'KRT6A', 'MTURN', 'GPX3', 'IGHA1', 'MEG3', 'MT-RNR2', 'HBA2', 'ACTB', 'CRYAB', 'SSU-rRNA', 'EPAS1', 'MT-ND2', 'HSPB7', 'IGHG3', 'NDRG2', 'CNN1', 'MT-CO1']]
2025-02-20 20:47:50,819 INFO intraExtraRNA.mapIds mapHUGO_2_ENSG() line:236] [BEGIN]
2025-02-20 20:48:23,750 INFO intraExtraRNA.mapIds mapHUGO_2_ENSG() line:266] [tx2GeneDF.shape : (5837318, 2)]
2025-02-20 20:48:23,752 INFO intraExtraRNA.mapIds mapHUGO_2_ENSG() line:267] [gene

In [14]:
print(f'elifeGenes :\n{elifeGenes}')
print(f'\nmissingElifeGenes :\n{missingElifeGenes}\n')
print(f'\nmapDF.shape : {mapDF.shape}' )
mapDF

elifeGenes :
['ENSG00000173641.18', 'ENSG00000018625.15', 'ENSG00000143248.13', 'ENSG00000232995.7', 'ENSG00000081277.13', 'ENSG00000163431.13', 'ENSG00000116016.14', 'ENSG00000168878.19', 'ENSG00000115414.21', 'ENSG00000175084.13', 'ENSG00000168309.18', 'ENSG00000189058.9', 'ENSG00000196616.14', 'ENSG00000211445.13', 'ENSG00000096696.15', 'ENSG00000244731.8', 'ENSG00000272398.6', 'ENSG00000112378.12', 'ENSG00000075624.17', 'ENSG00000180354.16', 'ENSG00000004799.8', 'ENSG00000120885.22', 'ENSG00000164687.11', 'ENSG00000198467.16', 'ENSG00000185303.17', 'ENSG00000095637.23', 'ENSG00000244734.4', 'ENSG00000167996.16', 'ENSG00000109846.9', 'ENSG00000137699.17', 'ENSG00000175899.15', 'ENSG00000205420.11', 'ENSG00000186081.12', 'ENSG00000155980.13', 'ENSG00000165795.25', 'ENSG00000196136.18', 'ENSG00000214548.18', 'ENSG00000211895.5', 'ENSG00000211897.9', 'ENSG00000182253.15', 'ENSG00000188536.13', 'ENSG00000206172.8', 'ENSG00000133392.18', 'ENSG00000128422.18', 'ENSG00000130176.8', 'ENSG00

,HUGO_v35,ENSG_v35,ENSG_v39
0,HSPB7,ENSG00000173641.17,ENSG00000173641.18
1,ATP1A2,ENSG00000018625.15,ENSG00000018625.15
2,RGS5,ENSG00000143248.13,ENSG00000143248.13
3,RGS5,ENSG00000232995.7,ENSG00000232995.7
4,PKP1,ENSG00000081277.13,ENSG00000081277.13
5,LMOD1,ENSG00000163431.13,ENSG00000163431.13
6,EPAS1,ENSG00000116016.14,ENSG00000116016.14
7,SFTPB,ENSG00000168878.19,ENSG00000168878.19
8,FN1,ENSG00000115414.20,ENSG00000115414.21
9,DES,ENSG00000175084.12,ENSG00000175084.13


# add column with the category type the gene came from
Note it is possible there are some genes that where duplicates. We do not record duplicates correctly

In [15]:
def createCategoryDF( geneSetDict : dict[str, pd.DataFrame] ) :
    retDF = pd.DataFrame()

    for category,geneNameDF in geneSetDict.items():
        tmpDF = geneNameDF.copy()
        tmpDF['category'] = [category] * geneNameDF.shape[0]
        retDF = pd.concat([retDF, tmpDF], ignore_index=True)

    return retDF

geneWithCategoryDF = createCategoryDF(geneSetDict)
print(f'geneWithCategoryDF.shape : {geneWithCategoryDF.shape}')
geneWithCategoryDF

geneWithCategoryDF.shape : (76, 2)


,name,category
0,DES,LUAD_vs_all
1,CLU,LUAD_vs_all
2,KRT6A,LUAD_vs_all
3,IGHA1,LUAD_vs_all
4,CRYAB,LUAD_vs_all
...,...,...
71,FN1,Whole_Blood_vs_all
72,MT-ND3,Whole_Blood_vs_all
73,FTH1,Whole_Blood_vs_all
74,FTL,Whole_Blood_vs_all


In [16]:
# def removeDups( df : pd.DataFrame) :
#     u = df.loc[:,"name"].unique()
#     selectUniqueRows = df.loc[:, "name"].isin( u )

#     return df.loc[selectUniqueRows, :]

# yyyDF = removeDups(xxxDF)
# print(f'yyyDF.shape : {yyyDF.shape}')
# yyyDF

In [17]:
mapDF = pd.merge(mapDF, geneWithCategoryDF, left_on='HUGO_v35', right_on='name', how='inner')
colIdx = 1
mapDF.drop('name', axis=colIdx, inplace=True)

In [19]:
print(f'mapDF.shape : {mapDF.shape}')
mapDF

mapDF.shape : (76, 4)


,HUGO_v35,ENSG_v35,ENSG_v39,category
0,HSPB7,ENSG00000173641.17,ENSG00000173641.18,LUSC_vs_all
1,HSPB7,ENSG00000173641.17,ENSG00000173641.18,Lung_vs_all
2,ATP1A2,ENSG00000018625.15,ENSG00000018625.15,LUAD_vs_all
3,RGS5,ENSG00000143248.13,ENSG00000143248.13,LUAD_vs_all
4,RGS5,ENSG00000143248.13,ENSG00000143248.13,LUSC_vs_all
...,...,...,...,...
71,MT-CO3,ENSG00000198938.2,ENSG00000198938.2,Whole_Blood_vs_all
72,MT-ND3,ENSG00000198840.2,ENSG00000198840.2,Whole_Blood_vs_all
73,MT-ND4,ENSG00000198886.2,ENSG00000198886.2,Whole_Blood_vs_all
74,MT-ND5,ENSG00000198786.2,ENSG00000198786.2,Whole_Blood_vs_all


In [20]:
# Find duplicates in 'col1'
duplicates = mapDF['HUGO_v35'].duplicated(keep=False)

# Print the boolean Series indicating duplicates
print(duplicates)
print("\n\n*****************\n")
# Filter the DataFrame to show only duplicate rows
duplicate_rows = mapDF[duplicates]
print(duplicate_rows)

0      True
1      True
2     False
3      True
4      True
      ...  
71    False
72    False
73    False
74    False
75    False
Name: HUGO_v35, Length: 76, dtype: bool


*****************

    HUGO_v35            ENSG_v35            ENSG_v39            category
0      HSPB7  ENSG00000173641.17  ENSG00000173641.18         LUSC_vs_all
1      HSPB7  ENSG00000173641.17  ENSG00000173641.18         Lung_vs_all
3       RGS5  ENSG00000143248.13  ENSG00000143248.13         LUAD_vs_all
4       RGS5  ENSG00000143248.13  ENSG00000143248.13         LUSC_vs_all
5       RGS5   ENSG00000232995.7   ENSG00000232995.7         LUAD_vs_all
6       RGS5   ENSG00000232995.7   ENSG00000232995.7         LUSC_vs_all
10     SFTPB  ENSG00000168878.19  ENSG00000168878.19         LUAD_vs_all
11     SFTPB  ENSG00000168878.19  ENSG00000168878.19         Lung_vs_all
13       DES  ENSG00000175084.12  ENSG00000175084.13         LUAD_vs_all
14       DES  ENSG00000175084.12  ENSG00000175084.13         LUSC_vs_all
15  

# save map file

In [22]:
path = f'{outDir}/mapDF.csv'
mapDF.to_csv( path, index= )
print(f'saved : {path}')

saved : /private/groups/kimlab/aedavids/deconvolution/tempus/hack-2025-02-20/AEDWIP_MAP.out/mapDF.csv
